In [0]:
-- Game - Play by Play
-- Every play in one game in order, with a one-line description. Includes the shootout.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2025020001 AS game_id,          -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
    CAST(NULL AS STRING) AS event   -- one play type, e.g. 'goal', 'hit', 'penalty', 'faceoff'; NULL = every play
),
game_players AS (                        -- everyone dressed in the game (game rosters)
  SELECT r.game_id, r.player_id, concat_ws(' ', r.first_name, r.last_name) AS full_name,
         r.position_code, r.sweater_number, r.team_id, t.abbrev AS team,
         r.team_id = g.home_team_id AS is_home, g.home_team_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.game_id = pr.game_id
),
plays AS (                               -- every play in the game, with the owning team's strength
  SELECT t.abbrev AS team,
         f.event_owner_team_id = g.home_team_id AS owner_is_home,
         IF(f.event_owner_team_id = g.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         CASE WHEN f.period_type IN ('OT', 'SO') THEN f.period_type
              ELSE CAST(f.period_number AS STRING) END AS period,
         SQRT(POW(89 - ABS(f.x_coord), 2) + POW(f.y_coord, 2)) AS distance_ft,
         f.*
  FROM nhl.gold.fact_event f
  JOIN nhl.gold.dim_game g ON f.game_id = g.game_id
  LEFT JOIN nhl.gold.dim_team t ON t.team_id = f.event_owner_team_id
  CROSS JOIN params pr
  WHERE f.game_id = pr.game_id
)
SELECT p.period,
       p.time_in_period          AS time,
       p.type_desc_key           AS event,
       p.team,
       CASE p.type_desc_key
         WHEN 'goal'         THEN concat_ws(' ', s.full_name,
                                   '(' || NULLIF(concat_ws(', ', a1.full_name, a2.full_name), '') || ')', p.shot_type)
         WHEN 'shot-on-goal' THEN concat_ws(' ', sh.full_name, p.shot_type, 'saved by', gl.full_name)
         WHEN 'missed-shot'  THEN concat_ws(' ', sh.full_name, p.shot_type, 'missed')
         WHEN 'blocked-shot' THEN concat_ws(' ', sh.full_name, 'blocked by', bl.full_name)
         WHEN 'hit'          THEN concat_ws(' ', ht.full_name, 'hit', he.full_name)
         WHEN 'faceoff'      THEN concat_ws(' ', fw.full_name, 'won vs', fl.full_name)
         WHEN 'penalty'      THEN concat_ws(' ', c.full_name, p.penalty_desc_key,
                                   p.penalty_duration || ' min', 'drawn by', d.full_name)
         WHEN 'takeaway'     THEN pl.full_name
         WHEN 'giveaway'     THEN pl.full_name
         ELSE p.stoppage_reason END AS description,
       p.strength,
       p.zone_code               AS zone,
       p.away_score,
       p.home_score,
       p.away_sog,
       p.home_sog
FROM plays p
LEFT JOIN game_players s  ON s.player_id  = p.scoring_player_id
LEFT JOIN game_players a1 ON a1.player_id = p.assist1_player_id
LEFT JOIN game_players a2 ON a2.player_id = p.assist2_player_id
LEFT JOIN game_players sh ON sh.player_id = p.shooting_player_id
LEFT JOIN game_players gl ON gl.player_id = p.goalie_in_net_id
LEFT JOIN game_players bl ON bl.player_id = p.blocking_player_id
LEFT JOIN game_players ht ON ht.player_id = p.hitting_player_id
LEFT JOIN game_players he ON he.player_id = p.hittee_player_id
LEFT JOIN game_players fw ON fw.player_id = p.winning_player_id
LEFT JOIN game_players fl ON fl.player_id = p.losing_player_id
LEFT JOIN game_players c  ON c.player_id  = p.committed_by_player_id
LEFT JOIN game_players d  ON d.player_id  = p.drawn_by_player_id
LEFT JOIN game_players pl ON pl.player_id = p.player_id
CROSS JOIN params pr
WHERE pr.event IS NULL OR p.type_desc_key = pr.event
ORDER BY p.period_number, p.sort_order;